# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/furkankumrudev/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
%pip -q install duckdb

In [2]:
import os
import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

print("DuckDB connection ready.")

DuckDB connection ready.


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row = one content item for one client on one report date.

I will use daily search performance data over a 90-day window, using March 2026 as the development month.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

REL = "hf://datasets/FlyRank/internship-warehouse"

fact_daily = f"""
read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')
"""

check = con.sql(f"""
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS min_date,
    MAX(report_date) AS max_date
FROM {fact_daily}
""").df()

check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,row_count,min_date,max_date
0,9841378,2026-03-01,2026-03-31


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Features (5 Features Max)
1. `gsc_impressions`: Measures search visibility on the report date.
   - *Knowable at the decision moment because it reflects past search impressions logged by Google Search Console before predicting future performance.*
2. `gsc_clicks`: Measures historical search traffic generated up to the decision moment.
   - *Knowable at the decision moment because it records user click interactions observed on that specific day.*
3. `has_gsc_position`: Binary indicator (`gsc_avg_position > 0`) of whether ranking telemetry was captured.
   - *Knowable at the decision moment because it is derived directly from contemporaneous Search Console position data.*
4. `gsc_ctr`: Click-through rate calculated as `clicks / impressions`.
   - *Knowable at the decision moment because it is computed entirely from contemporaneous daily metrics without future lookahead.*
5. `gsc_avg_position_clean`: Cleaned average position ranking (0 if unranked).
   - *Knowable at the decision moment because it reflects the observed search ranking logged on the report date.*

### Label / proxy
- `is_declining_label`: A proxy binary target indicating whether a content item experiences low search impressions compared to baseline performance.

### Context
- `client_hash_id`: Pseudonymized client identifier used for grouping and splits, never as a model feature.
- `content_hash_id`: Pseudonymized content identifier used for joins, never as a feature.
- `report_date`: Historical date stamp providing time context for sequential splitting.

### Excluded
- `trend_pct` / `trend_direction`: Directly derived from future/outcome windows. Excluded to strictly prevent label leakage.
- `ga4_*`: Excluded from this lane because GA4 integration history is absent or zero-filled across several clients.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Check the fields used in the contract
field_check = con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(gsc_impressions) AS impressions_available,
    COUNT(gsc_clicks) AS clicks_available,
    COUNT(gsc_avg_position) AS position_available,
    COUNT(DISTINCT client_hash_id) AS clients,
    COUNT(DISTINCT content_hash_id) AS content_items
FROM {fact_daily}
WHERE report_date >= DATE '2026-03-01'
  AND report_date < DATE '2026-04-01'
""").df()

print("Field availability and context check for March 2026:")
display(field_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Field availability and context check for March 2026:


,total_rows,impressions_available,clicks_available,position_available,clients,content_items
0,9841378,9841378,9841378,3611061,55,331437


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
import numpy as np

# ==========================================
# 1. THREE VERIFICATION QUERIES
# ==========================================

# Fact 1: Grain Check (Empty DataFrame proves grain holds: report_date x client x content)
grain_check = con.sql(f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    COUNT(*) AS n
FROM {fact_daily}
GROUP BY 1, 2, 3
HAVING COUNT(*) > 1
LIMIT 5
""").df()

print("Fact 1 — Duplicate grain rows (must return empty):")
display(grain_check)


# Fact 2: Slice Row Count and Date Span
count_check = con.sql(f"""
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS min_date,
    MAX(report_date) AS max_date
FROM {fact_daily}
""").df()

print("Fact 2 — March 2026 row count and date span:")
display(count_check)


# Fact 3: Availability Check (using IS TRUE)
availability_check = con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) FILTER (
        WHERE gsc_data_available IS TRUE
    ) AS available_rows,
    COUNT(*) FILTER (
        WHERE gsc_data_available IS TRUE
    ) * 100.0 / COUNT(*) AS available_pct
FROM {fact_daily}
""").df()

print("Fact 3 — GSC availability filtered with IS TRUE:")
display(availability_check)


# ==========================================
# 2. FIVE-FEATURE FRAME (March 2026 slice)
# ==========================================

feature_frame = con.sql(f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    gsc_impressions,
    gsc_clicks,
    CASE WHEN gsc_avg_position > 0 THEN 1 ELSE 0 END AS has_gsc_position,
    CASE WHEN gsc_impressions > 0 THEN (gsc_clicks * 1.0 / gsc_impressions) ELSE 0.0 END AS gsc_ctr,
    COALESCE(gsc_avg_position, 0.0) AS gsc_avg_position_clean
FROM {fact_daily}
WHERE gsc_data_available IS TRUE
LIMIT 10000
""").df()

print("\n--- 5-Feature Frame Sample (10,000 rows) ---")
display(feature_frame.head())


# ==========================================
# 3. THE LEAKAGE TRAP EXPERIMENT
# ==========================================

df_exp = feature_frame.copy()

# Proxy Label: Content performing at or below median impressions (1 = declining/low, 0 = active)
median_imp = df_exp['gsc_impressions'].median()
df_exp['is_declining_label'] = (df_exp['gsc_impressions'] <= median_imp).astype(int)

# --- 3a. The Trap: Deliberate Target Leak ---
df_exp['leaked_target_proxy'] = df_exp['is_declining_label'] * 0.95 + np.random.normal(0, 0.01, size=len(df_exp))

features_leaked = ['gsc_clicks', 'has_gsc_position', 'gsc_ctr', 'gsc_avg_position_clean', 'leaked_target_proxy']
X_train_l, X_test_l, y_train_l, y_test_l = train_test_split(
    df_exp[features_leaked], df_exp['is_declining_label'], test_size=0.3, random_state=42, stratify=df_exp['is_declining_label']
)

model_leaked = RandomForestClassifier(max_depth=3, random_state=42)
model_leaked.fit(X_train_l, y_train_l)
auc_leaked = roc_auc_score(y_test_l, model_leaked.predict_proba(X_test_l)[:, 1])
print(f"\n🚨 [THE TRAP] Leaked Model ROC-AUC: {auc_leaked:.4f} (Artificially inflated due to leaked feature)")

# --- 3b. The Honest Model: Leak removed, honest baseline retained ---
features_honest = ['gsc_clicks', 'has_gsc_position', 'gsc_ctr', 'gsc_avg_position_clean']
X_train_h, X_test_h, y_train_h, y_test_h = train_test_split(
    df_exp[features_honest], df_exp['is_declining_label'], test_size=0.3, random_state=42, stratify=df_exp['is_declining_label']
)

model_honest = RandomForestClassifier(max_depth=3, random_state=42)
model_honest.fit(X_train_h, y_train_h)
auc_honest = roc_auc_score(y_test_h, model_honest.predict_proba(X_test_h)[:, 1])
print(f"✅ [HONEST MODEL] Non-leaked ROC-AUC: {auc_honest:.4f} (True decision-time baseline)")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Fact 1 — Duplicate grain rows (must return empty):


,report_date,client_hash_id,content_hash_id,n


Fact 2 — March 2026 row count and date span:


,row_count,min_date,max_date
0,9841378,2026-03-01,2026-03-31


...

Fact 3 — GSC availability filtered with IS TRUE:


,total_rows,available_rows,available_pct
0,9841378,3611061,36.692636



--- 5-Feature Frame Sample (10,000 rows) ---


,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,has_gsc_position,gsc_ctr,gsc_avg_position_clean
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,20,0,1,0.000,3.350000
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,1,0,0,0.000,0.000000
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,125,1,1,0.008,4.928000
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,7,0,1,0.000,4.000000
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,11,0,1,0.000,2.272727



🚨 [THE TRAP] Leaked Model ROC-AUC: 1.0000 (Artificially inflated due to leaked feature)
✅ [HONEST MODEL] Non-leaked ROC-AUC: 0.7789 (True decision-time baseline)


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

### Data limits

This dataset cannot give a perfectly balanced view of all clients because history depth differs between clients. Some clients have much less search history than others.

The data also cannot treat all zero values as true zero performance. In particular, availability flags must be checked before interpreting missing or zero-filled data.

The final month should not be used to develop the label logic because it is the natural outcome window. I use March 2026 as the development month and keep the final month sealed for later evaluation.

Another limitation is that this analysis focuses on Search Console performance, so it does not fully explain business outcomes such as conversions or revenue.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Check the main data limitations described above

# 1. Client history depth
history_check = con.sql(f"""
SELECT
    COUNT(DISTINCT client_hash_id) AS clients,
    COUNT(DISTINCT CASE
        WHEN report_date <= DATE '2026-03-01' - INTERVAL 365 DAY
        THEN client_hash_id
    END) AS clients_with_12m_history
FROM {fact_daily}
""").df()

print("Client history depth:")
display(history_check)


# 2. GSC-only / unavailable rows
availability_check = con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) FILTER (
        WHERE gsc_data_available IS TRUE
    ) AS gsc_available_rows,
    COUNT(*) FILTER (
        WHERE gsc_data_available IS NOT TRUE
    ) AS gsc_unavailable_rows
FROM {fact_daily}
WHERE report_date >= DATE '2026-03-01'
  AND report_date < DATE '2026-04-01'
""").df()

print("GSC availability limitation:")
display(availability_check)


# 3. Final month is separated from the development month
window_check = con.sql(f"""
SELECT
    MIN(report_date) AS min_date,
    MAX(report_date) AS max_date,
    COUNT(DISTINCT report_date) AS distinct_dates
FROM {fact_daily}
""").df()

print("Available date range:")
display(window_check)

Client history depth:


,clients,clients_with_12m_history
0,55,0


GSC availability limitation:


,total_rows,gsc_available_rows,gsc_unavailable_rows
0,9841378,3611061,6230317


Available date range:


,min_date,max_date,distinct_dates
0,2026-03-01,2026-03-31,31


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.